In [ ]:
# Setup

import numpy as np
import pandas as pd

from blox2 import SteinNoveltySelector, RandomSelector, split_df_by_n_rows, load_features
from blox2.predictor import RidgePredictor, LassoPredictor, RandomForestPredictor, SVRPredictor

# predictor = RidgePredictor()
# predictor = LassoPredictor()
predictor = RandomForestPredictor(n_estimators=100)
# predictor = SVRPredictor()

observed_features, unchecked_features = split_df_by_n_rows(load_features("../data/zinc_dft/feature_list_mfp2048_rdd.npz"), 10)
observed_values, unchecked_values = split_df_by_n_rows(pd.read_csv("../data/zinc_dft/properties.csv"), 10)

# observed_features = pd.read_csv("../data/test/feature_list_of_observed_data.csv", header=None)
# unchecked_features = pd.read_csv("../data/test/feature_list_of_unchecked_data.csv", header=None)
# observed_values = pd.read_csv("../data/test/properties_of_observed_data.csv")
# unchecked_values = pd.read_csv("../data/test/properties_of_unchecked_data.csv")

def get_true_value(id):
    return np.asarray(unchecked_values.iloc[id - len(observed_features)])

In [ ]:
# Loop

import time

n_iters = 2000
report_interval = 50
sigma = 1

selector = SteinNoveltySelector(observed_features, observed_values, unchecked_features, predictor, sigma=sigma, normalize_features=False, n_obs_samples=None, value_normalization="after_pred", chunk_size=512, use_distribution=False, compare_selection_time=False)
# selector = RandomSelector(observed_features, observed_values, unchecked_features)

t0 = time.perf_counter()
for i in range(min(n_iters, len(unchecked_features))):
    ids = selector.next_candidates(n=1)
    for id in ids:
        selector.observe(id, get_true_value(id))
    if (i+1) % report_interval == 0:
        print(f"{i+1} candidates suggested. Passed time: {time.perf_counter() - t0}")

In [ ]:
import csv, datetime, os
import matplotlib.pyplot as plt
import pandas as pd
from blox2 import calc_stein_discrepancy_trajectory

fixed_data_for_scaler = np.vstack([observed_values, unchecked_values]) # uses all property values
scatter_plot_intervals = [200, 500, 2000, 5000, 20000, 50000]
dir_name = "results/" + datetime.datetime.now().strftime("%m-%d_%H-%M")

In [ ]:
# Record results
os.makedirs(dir_name, exist_ok=True)
observation_history = selector.make_observation_history()

with open(os.path.join(dir_name, "candidate_histories.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    for x in selector.candidate_id_history[selector.initial_n_obs:]:
        writer.writerow([x])
np.savetxt(os.path.join(dir_name, "initial_observed_properties.csv"), observation_history[:selector.initial_n_obs], delimiter=",",)
np.savetxt(os.path.join(dir_name, "observation_histories.csv"), observation_history[selector.initial_n_obs:], delimiter=",",)

In [ ]:
# Record times

df = pd.DataFrame({
    "Selection": selector.passed_times_selection,
    "Train": selector.passed_times_train,
    "Pred": selector.passed_times_pred,
    "Total": selector.passed_times_total
})
df["Misc"] = df["Total"] - df["Selection"] - df["Train"] - df["Pred"]
df.to_csv(os.path.join(dir_name, "time_consumption.csv"), index=False)
ax = df.drop(columns="Total").plot()
ax.set_xlabel("Number of samplings")
ax.set_ylabel("Time consumption (sec)")
plt.tight_layout()
plt.savefig(os.path.join(dir_name, "time_consumption.png"))
plt.show()

In [ ]:
# Plot and save Stein discrepancy trajectory with fixed scaling

sd_trajectory = calc_stein_discrepancy_trajectory(observation_history, scale=fixed_data_for_scaler, sigma=sigma)

y = sd_trajectory[selector.initial_n_obs:]
x = np.arange(10, 10 + len(y))
plt.plot(x, y)
plt.xscale("log", base=10)
plt.xticks([10, 100, 1000], [r"$10^1$", r"$10^2$", r"$10^3$"])
plt.xlabel("Number of samplings")
plt.ylabel("Stein discrepancy")
plt.grid(True, axis="y")
plt.savefig(os.path.join(dir_name, f"stein_discrepancy_s={sigma}.png"))
plt.show()
np.savetxt(os.path.join(dir_name, f"stein_discrepancy_histories_s={sigma}.csv"), sd_trajectory[selector.initial_n_obs:], delimiter=",",)

In [ ]:
# Scatterplot

for n in scatter_plot_intervals:
    if n <= 0 or n > len(observation_history):
        continue
    data = observation_history[:n + selector.initial_n_obs + 1]

    plt.figure()
    plt.scatter(data[:, 0], data[:, 1], s=5)
    plt.xlabel(observed_values.columns[0])
    plt.ylabel(observed_values.columns[1])
    plt.title(f"Number of sampling: {n}")
    plt.savefig(os.path.join(dir_name, f"scatter_{n}.png"))
    plt.grid(True)
    plt.show()